## 1. Data
Simulated ED patients with continuous vitals and a known true model.

### 1.1 Setup

In [1]:
import os

# must be set before pymc/jax are imported
os.environ["JAX_PLATFORMS"] = "cpu"                                     # GPU was slower for this model
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=6"    # one JAX device per chain, run in parallel

import numpy as np
import pandas as pd
import pymc as pm

SEED = 42

### 1.2 Patients

In [2]:
rng = np.random.default_rng(SEED)
N_TRAIN = 8_000
N_TEST = 2_000                   # 80/20 split
N = N_TRAIN + N_TEST
COMPLAINTS = ["chest pain", "shortness of breath", "headache", "abdominal pain"]

df = pd.DataFrame({
    "age": rng.normal(55, 18, N).clip(18, 100).round(),
    "male": rng.binomial(1, 0.5, N),
    "complaint": rng.choice(COMPLAINTS, N),
})
# vital signs, shaped after the real ED patients in data.csv.xls
fever = rng.random(N) < 0.10                      # 10% febrile
cold = rng.random(N) < 0.005                      # 0.5% hypothermic
V1 = rng.normal(36.55, 0.4, N) + fever * rng.gamma(2.0, 0.6, N)
df["V1"] = np.where(cold, rng.normal(34.6, 0.6, N), V1).clip(30, 42.5).round(1)                   # body temperature, °C
df["V2"] = (rng.lognormal(np.log(131), 0.2, N) + 0.4 * (df["age"] - 55)).clip(50, 260).round()     # systolic BP, mmHg
df["V3"] = (rng.lognormal(np.log(82), 0.21, N) + 10 * (df["V1"] - 36.6)).clip(30, 220).round()    # heart rate, bpm

# random train/test split
df["split"] = np.where(rng.permutation(N) < N_TRAIN, "train", "test")
df.head()

,age,male,complaint,V1,V2,V3,split
0,60.0,1,chest pain,36.0,110.0,68.0,train
1,36.0,0,chest pain,36.5,131.0,108.0,train
2,69.0,1,abdominal pain,36.1,202.0,69.0,train
3,72.0,0,headache,38.9,144.0,87.0,train
4,20.0,0,abdominal pain,36.5,97.0,70.0,train


### 1.3 Features (standardised, vitals + squares)

In [3]:
train = df["split"] == "train"
test = ~train
# standardise with training statistics only, so nothing about the test set leaks into the fit
AGE_MEAN, AGE_SD = df.loc[train, "age"].mean(), df.loc[train, "age"].std()
VITALS = ["V1", "V2", "V3"]
VITAL_MEAN, VITAL_SD = df.loc[train, VITALS].mean(), df.loc[train, VITALS].std()

X = pd.DataFrame({
    "age": (df["age"] - AGE_MEAN) / AGE_SD,
    "male": df["male"],
})
for c in COMPLAINTS[1:]:
    X[c] = (df["complaint"] == c).astype(int)
# each vital stays continuous: a standardised linear term plus its square, so both low and high values can raise risk
for v in VITALS:
    z = (df[v] - VITAL_MEAN[v]) / VITAL_SD[v]
    X[v] = z
    X[v + "_sq"] = z ** 2
X.head()

,age,male,shortness of breath,headache,abdominal pain,V1,V1_sq,V2,V2_sq,V3,V3_sq
0,0.279896,1,0,0,0,-1.057196,1.117664,-0.838434,0.702971,-0.842850,0.710396
1,-1.085727,0,0,0,0,-0.237124,0.056228,-0.084156,0.007082,1.276111,1.628459
2,0.792005,1,0,0,1,-0.893182,0.797773,2.466021,6.081260,-0.789876,0.623904
3,0.962707,0,0,1,0,3.699223,13.684248,0.382778,0.146519,0.163657,0.026783
4,-1.996143,0,0,0,1,-0.237124,0.056228,-1.305367,1.703984,-0.736902,0.543024


### 1.4 True model and deaths

In [4]:
# the true values PyMC should recover
rng = np.random.default_rng(SEED)
TRUE_INTERCEPT = -4.2           # keeps mortality near 3% with the vitals added
TRUE_BETA = pd.Series({
    "age": 0.8,                   # per standard deviation of age (about 18 years)
    "male": 0.3,
    "shortness of breath": 0.5,
    "headache": -0.7,
    "abdominal pain": -0.2,
    # vitals, per standard deviation (linear) and per squared standard deviation (curvature)
    "V1": 0.1,   "V1_sq": 0.04,     # temperature: long fever tail, so a small square term
    "V2": -0.3,  "V2_sq": 0.1,      # systolic BP: low is worse than high
    "V3": 0.2,   "V3_sq": 0.1,      # heart rate
})

p_true = 1 / (1 + np.exp(-(TRUE_INTERCEPT + X @ TRUE_BETA)))
df["dead"] = rng.binomial(1, p_true)
print(f"{df['dead'].sum()} deaths of {N} ({df['dead'].mean():.1%})")
df.groupby("split")["dead"].agg(patients="size", deaths="sum", rate="mean")

328 deaths of 10000 (3.3%)


,patients,deaths,rate
split,,,
test,2000,63,0.031500
train,8000,265,0.033125


### 1.5 Training batches

In [5]:
# each training patient gets one of 4 batches; the updating steps each get their own batch
df["batch"] = 0
df.loc[train, "batch"] = np.arange(train.sum()) % 4 + 1

# one row per training patient, no collapsing
X_train = X[train]
y_train = df.loc[train, "dead"].values
batch_train = df.loc[train, "batch"].values
print(f"{len(X_train)} training patients, {y_train.sum()} deaths")

8000 training patients, 265 deaths


## 2. Models
Logistic regression, N(0, 1) priors, one row per patient.

### 2.1 All features, all training data

In [ ]:
with pm.Model(coords={"feature": X.columns}) as model:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta = pm.Normal("beta", mu=0, sigma=1, dims="feature")

    pm.Bernoulli("dead", logit_p=intercept + pm.math.dot(X_train.values, beta), observed=y_train)

    # 6 chains x 667 draws ≈ 4000 draws, sampled with NumPyro (JAX) on the CPU
    idata = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

### 2.2 Batch 1 base features

In [26]:
BASE_COLS = ["age", "male"] + COMPLAINTS[1:]
rows = batch_train == 1
X_train.loc[rows, BASE_COLS]

,age,male,shortness of breath,headache,abdominal pain
0,0.279896,1,0,0,0
4,-1.996143,0,0,0,1
8,-0.004609,0,0,1,0
12,0.052292,0,0,0,1
18,0.905807,1,0,0,1
...,...,...,...,...,...
9979,-1.085727,1,0,0,0
9983,-0.004609,1,0,0,1
9988,0.735104,0,0,1,0
9992,-0.573619,1,1,0,0


### 2.3 Update 1: base (age, sex, complaint), batch 1

In [ ]:
BASE_COLS = ["age", "male"] + COMPLAINTS[1:] # use columns age, gender, complaint
rows = batch_train == 1

with pm.Model(coords={"feature": BASE_COLS}) as model_base:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta = pm.Normal("beta", mu=0, sigma=1, dims="feature")

    pm.Bernoulli("dead", logit_p=intercept + pm.math.dot(X_train.loc[rows, BASE_COLS].values, beta),
                 observed=y_train[rows])

    idata_base = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [intercept, beta]
Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]



















Running chain 0:   0%|          | 0/1667 [00:01<?, ?it/s]















Running chain 0:  10%|▉         | 166/1667 [00:01<00:01, 1042.05it/s]









Running chain 0:  20%|█▉        | 332/1667 [00:01<00:01, 1063.20it/s]








Running chain 0:  30%|██▉       | 498/1667 [00:01<00:01, 1119.39it/s]









Running chain 0:  40%|███▉      | 664/1667 [00:01<00:00, 1200.09it/s]













Running chain 0:  50%|████▉     | 830/1667 [00:02<00:00, 1167.58it/s]













Running chain 0:  60%|█████▉    | 996/1667 [00:02<00:00, 1134.64it/s]







Running chain 0:  70%|██████▉   | 1162/1667 [00:02<00:00, 1084.90it/s]








Running chain 0:  80%|███████▉  | 1328/1667 [00:02<00:00, 1141.31it/s]









Running chain 0:  90%|████████▉ | 1494/1667 [00:02<00:00, 1115.24it/s]





Running chain 0: 100%|██████████| 1667/1667 [00:02<00:00, 587.40it/s] 




Running chain 5: 100%|█████████

### 2.4 Update 2: + V1 temperature, batch 2

In [9]:
COLS_V1 = BASE_COLS + ["V1", "V1_sq"]
rows = batch_train == 2

# prior for the parameters the previous step already has: mean and covariance of its posterior draws
prev = np.column_stack([idata_base.posterior["intercept"].values.ravel(),
                        idata_base.posterior["beta"].values.reshape(-1, len(BASE_COLS))])

with pm.Model(coords={"feature": COLS_V1}) as model_v1:
    old = pm.MvNormal("old", mu=prev.mean(axis=0), cov=np.cov(prev, rowvar=False))     # intercept + BASE_COLS
    new = pm.Normal("new", mu=0, sigma=1, shape=len(COLS_V1) - len(BASE_COLS))   # the added vital
    intercept = pm.Deterministic("intercept", old[0])
    beta = pm.Deterministic("beta", pm.math.concatenate([old[1:], new]), dims="feature")

    pm.Bernoulli("dead", logit_p=intercept + pm.math.dot(X_train.loc[rows, COLS_V1].values, beta),
                 observed=y_train[rows])

    idata_v1 = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [old, new]
Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]



















Running chain 0:   0%|          | 0/1667 [00:01<?, ?it/s]









Running chain 0:   5%|▍         | 83/1667 [00:01<00:04, 349.72it/s]









Running chain 0:  10%|▉         | 166/1667 [00:01<00:03, 441.98it/s]












Running chain 0:  15%|█▍        | 249/1667 [00:02<00:02, 514.00it/s]









Running chain 0:  25%|██▍       | 415/1667 [00:02<00:01, 706.47it/s]


Running chain 0:  35%|███▍      | 581/1667 [00:02<00:01, 816.11it/s]












Running chain 0:  45%|████▍     | 747/1667 [00:02<00:00, 939.79it/s]









Running chain 0:  55%|█████▍    | 913/1667 [00:02<00:00, 966.70it/s]
















Running chain 0:  65%|██████▍   | 1079/1667 [00:02<00:00, 833.09it/s]









Running chain 0:  75%|███████▍  | 1245/1667 [00:03<00:00, 822.29it/s]








Running chain 0:  85%|████████▍ | 1411/1667 [00:03<00:00, 879.73it/s]







Running chain 0:  95%|█████████▍| 1577/1667 [00:

### 2.5 Update 3: + V2 blood pressure, batch 3

In [10]:
COLS_V2 = COLS_V1 + ["V2", "V2_sq"]
rows = batch_train == 3

# prior for the parameters the previous step already has: mean and covariance of its posterior draws
prev = np.column_stack([idata_v1.posterior["intercept"].values.ravel(),
                        idata_v1.posterior["beta"].values.reshape(-1, len(COLS_V1))])

with pm.Model(coords={"feature": COLS_V2}) as model_v2:
    old = pm.MvNormal("old", mu=prev.mean(axis=0), cov=np.cov(prev, rowvar=False))     # intercept + COLS_V1
    new = pm.Normal("new", mu=0, sigma=1, shape=len(COLS_V2) - len(COLS_V1))   # the added vital
    intercept = pm.Deterministic("intercept", old[0])
    beta = pm.Deterministic("beta", pm.math.concatenate([old[1:], new]), dims="feature")

    pm.Bernoulli("dead", logit_p=intercept + pm.math.dot(X_train.loc[rows, COLS_V2].values, beta),
                 observed=y_train[rows])

    idata_v2 = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [old, new]
Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]



















Running chain 0:   0%|          | 0/1667 [00:01<?, ?it/s]


















Running chain 0:   5%|▍         | 83/1667 [00:01<00:06, 226.92it/s]











Running chain 0:  10%|▉         | 166/1667 [00:02<00:05, 261.59it/s]








Running chain 0:  15%|█▍        | 249/1667 [00:02<00:04, 353.33it/s]





Running chain 0:  20%|█▉        | 332/1667 [00:02<00:03, 418.95it/s]














Running chain 0:  30%|██▉       | 498/1667 [00:02<00:02, 568.78it/s]









Running chain 0:  40%|███▉      | 664/1667 [00:02<00:01, 704.18it/s]












Running chain 0:  50%|████▉     | 830/1667 [00:02<00:01, 805.17it/s]







Running chain 0:  60%|█████▉    | 996/1667 [00:03<00:00, 875.39it/s]









Running chain 0:  70%|██████▉   | 1162/1667 [00:03<00:00, 866.96it/s]









Running chain 1: 100%|██████████| 1667/1667 [00:03<00:00, 481.37it/s]


Running chain 0:  80%|███████▉  | 1328/1667 [00

### 2.6 Update 4: + V3 heart rate, batch 4

In [11]:
COLS_V3 = COLS_V2 + ["V3", "V3_sq"]
rows = batch_train == 4

# prior for the parameters the previous step already has: mean and covariance of its posterior draws
prev = np.column_stack([idata_v2.posterior["intercept"].values.ravel(),
                        idata_v2.posterior["beta"].values.reshape(-1, len(COLS_V2))])

with pm.Model(coords={"feature": COLS_V3}) as model_v3:
    old = pm.MvNormal("old", mu=prev.mean(axis=0), cov=np.cov(prev, rowvar=False))     # intercept + COLS_V2
    new = pm.Normal("new", mu=0, sigma=1, shape=len(COLS_V3) - len(COLS_V2))   # the added vital
    intercept = pm.Deterministic("intercept", old[0])
    beta = pm.Deterministic("beta", pm.math.concatenate([old[1:], new]), dims="feature")

    pm.Bernoulli("dead", logit_p=intercept + pm.math.dot(X_train.loc[rows, COLS_V3].values, beta),
                 observed=y_train[rows])

    idata_v3 = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [old, new]
Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]



















Running chain 0:   0%|          | 0/1667 [00:01<?, ?it/s]















Running chain 0:   5%|▍         | 83/1667 [00:01<00:07, 205.54it/s]







Running chain 0:  10%|▉         | 166/1667 [00:02<00:05, 284.58it/s]








Running chain 0:  15%|█▍        | 249/1667 [00:02<00:03, 396.32it/s]









Running chain 0:  20%|█▉        | 332/1667 [00:02<00:02, 458.38it/s]









Running chain 0:  25%|██▍       | 415/1667 [00:02<00:02, 481.81it/s]





Running chain 0:  30%|██▉       | 498/1667 [00:02<00:02, 504.49it/s]





Running chain 0:  35%|███▍      | 581/1667 [00:02<00:02, 530.10it/s]









Running chain 0:  40%|███▉      | 664/1667 [00:02<00:01, 572.45it/s]



















Running chain 0:  50%|████▉     | 830/1667 [00:03<00:01, 708.03it/s]


Running chain 0:  55%|█████▍    | 913/1667 [00:03<00:01, 733.97it/s]






Running chain 0:  60%|█████▉    | 996/1667 [00:03<00:00, 73

### 2.7 Comparing the steps

In [12]:
def draws_of(idata, cols):
    """Posterior draws as a table: one column for the intercept and one per feature."""
    return pd.DataFrame(np.column_stack([idata.posterior["intercept"].values.ravel(),
                                         idata.posterior["beta"].values.reshape(-1, len(cols))]),
                        columns=["intercept"] + cols)

models = {
    "all at once": draws_of(idata, list(X.columns)),
    "base": draws_of(idata_base, BASE_COLS),
    "+V1": draws_of(idata_v1, COLS_V1),
    "+V2": draws_of(idata_v2, COLS_V2),
    "+V3": draws_of(idata_v3, COLS_V3),
}

table = pd.DataFrame({"true": pd.Series({"intercept": TRUE_INTERCEPT, **TRUE_BETA})})
for name, d in models.items():
    table[name] = d.mean()
table["sd all at once"] = models["all at once"].std()
table["sd +V3"] = models["+V3"].std()
table.round(2)

,true,all at once,base,+V1,+V2,+V3,sd all at once,sd +V3
intercept,-4.20,-4.07,-3.46,-3.67,-3.88,-3.91,0.16,0.15
age,0.80,0.87,0.84,0.89,0.88,0.85,0.07,0.07
male,0.30,0.32,-0.07,0.13,0.25,0.32,0.13,0.13
shortness of breath,0.50,0.42,0.16,0.43,0.52,0.45,0.16,0.16
headache,-0.70,-0.79,-1.57,-0.86,-0.78,-0.74,0.22,0.22
abdominal pain,-0.20,-0.15,-0.22,-0.36,-0.28,-0.14,0.18,0.19
V1,0.10,0.14,NaN,0.12,0.12,0.17,0.08,0.08
V1_sq,0.04,0.03,NaN,0.04,0.06,0.04,0.02,0.02
V2,-0.30,-0.23,NaN,NaN,-0.21,-0.33,0.07,0.10
V2_sq,0.10,0.10,NaN,NaN,0.07,0.08,0.04,0.05


## 3. Posterior vs truth
95% intervals of model 2.1 against the true values.

### 3.1 Intervals

In [13]:
draws = {"intercept": idata.posterior["intercept"].values.ravel()}
beta_draws = idata.posterior["beta"].values.reshape(-1, X.shape[1])
for j, name in enumerate(X.columns):
    draws[name] = beta_draws[:, j]

truth = {"intercept": TRUE_INTERCEPT, **TRUE_BETA}
result = pd.DataFrame({
    "true": truth,
    "mean": {k: v.mean() for k, v in draws.items()},
    "2.5%": {k: np.quantile(v, 0.025) for k, v in draws.items()},
    "97.5%": {k: np.quantile(v, 0.975) for k, v in draws.items()},
})
result["inside"] = (result["2.5%"] <= result["true"]) & (result["true"] <= result["97.5%"])
result.round(2)

,true,mean,2.5%,97.5%,inside
intercept,-4.20,-4.07,-4.41,-3.77,True
age,0.80,0.87,0.73,1.00,True
male,0.30,0.32,0.07,0.57,True
shortness of breath,0.50,0.42,0.11,0.74,True
headache,-0.70,-0.79,-1.21,-0.37,True
abdominal pain,-0.20,-0.15,-0.51,0.19,True
V1,0.10,0.14,-0.01,0.29,True
V1_sq,0.04,0.03,-0.00,0.06,True
V2,-0.30,-0.23,-0.37,-0.09,True
V2_sq,0.10,0.10,0.02,0.17,True


## 4. Predict for one patient
70-year-old man, shortness of breath, normal vitals.

### 4.1 Prediction

In [14]:
# a 70-year-old man with shortness of breath and normal vitals
patient = pd.Series(0.0, index=X.columns)
patient["age"] = (70 - AGE_MEAN) / AGE_SD
patient["male"] = 1
patient["shortness of breath"] = 1
for v, value in {"V1": 36.6, "V2": 130, "V3": 80}.items():
    z = (value - VITAL_MEAN[v]) / VITAL_SD[v]
    patient[v], patient[v + "_sq"] = z, z ** 2

logit_draws = draws["intercept"] + beta_draws @ patient.values
p_draws = 1 / (1 + np.exp(-logit_draws))
p_true_patient = 1 / (1 + np.exp(-(TRUE_INTERCEPT + patient @ TRUE_BETA)))

print(f"true P(death):      {p_true_patient:.3f}")
print(f"posterior mean:     {p_draws.mean():.3f}")
print(f"95% interval:       {np.quantile(p_draws, 0.025):.3f} – {np.quantile(p_draws, 0.975):.3f}")

true P(death):      0.061
posterior mean:     0.068
95% interval:       0.052 – 0.086


## 5. Test set
AUC, Brier score and calibration on unseen patients.

### 5.1 Helpers

In [15]:
X_test = X[test]
y_test = df.loc[test, "dead"].values
p_true_test = p_true[test].values
print(f"test patients: {len(y_test)}, deaths: {y_test.sum()}")

def predict_test(d):
    """Posterior mean P(death) per test patient; d = draws with an intercept column and one column per feature."""
    cols = list(d.columns[1:])
    Xt = X_test[cols].values
    a, b = d["intercept"].values, d[cols].values
    p_sum = np.zeros(len(Xt))
    for s in range(0, len(a), 250):                      # in chunks of draws to save memory
        p_sum += (1 / (1 + np.exp(-(a[s:s + 250, None] + b[s:s + 250] @ Xt.T)))).sum(axis=0)
    return p_sum / len(a)

def auc(y, p):
    """Probability that a random death gets a higher prediction than a random survivor."""
    r = pd.Series(p).rank().values
    n1, n0 = y.sum(), len(y) - y.sum()
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def evaluate(d):
    """Print AUC and Brier score for the model with draws d, and return its calibration table."""
    p = predict_test(d)
    print(f"AUC          model {auc(y_test, p):.3f}    true probabilities {auc(y_test, p_true_test):.3f}")
    print(f"Brier score  model {np.mean((p - y_test) ** 2):.4f}   true probabilities {np.mean((p_true_test - y_test) ** 2):.4f}")
    print(f"mean |p_model - p_true|: {np.abs(p - p_true_test).mean():.4f}")

    cal = pd.DataFrame({"predicted": p, "observed": y_test, "true": p_true_test})
    cal["risk tenth"] = pd.qcut(cal["predicted"], 10, labels=False, duplicates="drop") + 1
    return cal.groupby("risk tenth").agg(patients=("observed", "size"), predicted=("predicted", "mean"),
                                         observed=("observed", "mean"), true=("true", "mean")).round(3)

test patients: 2000, deaths: 63


### 5.2 Model 2.1: all at once

In [18]:
evaluate(models["all at once"])

AUC          model 0.679    true probabilities 0.682
Brier score  model 0.0301   true probabilities 0.0301
mean |p_model - p_true|: 0.0048


,patients,predicted,observed,true
risk tenth,,,,
1,200,0.004,0.010,0.004
2,200,0.007,0.005,0.007
3,200,0.011,0.015,0.010
4,200,0.014,0.035,0.013
5,200,0.018,0.020,0.017
6,200,0.023,0.025,0.021
7,200,0.030,0.040,0.028
8,200,0.042,0.030,0.038
9,200,0.062,0.055,0.054


### 5.3 Model 2.3: base

In [16]:
evaluate(models["base"])

AUC          model 0.606    true probabilities 0.682
Brier score  model 0.0305   true probabilities 0.0301
mean |p_model - p_true|: 0.0157


,patients,predicted,observed,true
risk tenth,,,,
1,202,0.003,0.015,0.006
2,199,0.007,0.025,0.010
3,201,0.011,0.030,0.017
4,201,0.014,0.030,0.017
5,198,0.019,0.025,0.024
6,204,0.024,0.020,0.025
7,195,0.031,0.041,0.029
8,203,0.043,0.020,0.045
9,198,0.060,0.035,0.054


### 5.4 Model 2.4: + V1

In [17]:
evaluate(models["+V1"])

AUC          model 0.639    true probabilities 0.682
Brier score  model 0.0308   true probabilities 0.0301
mean |p_model - p_true|: 0.0144


,patients,predicted,observed,true
risk tenth,,,,
1,200,0.004,0.010,0.005
2,200,0.008,0.020,0.009
3,200,0.011,0.015,0.012
4,200,0.015,0.035,0.015
5,200,0.020,0.020,0.020
6,200,0.026,0.040,0.023
7,201,0.035,0.025,0.030
8,199,0.047,0.035,0.041
9,200,0.069,0.040,0.055


### 5.5 Model 2.5: + V2

In [19]:
evaluate(models["+V2"])

AUC          model 0.664    true probabilities 0.682
Brier score  model 0.0306   true probabilities 0.0301
mean |p_model - p_true|: 0.0117


,patients,predicted,observed,true
risk tenth,,,,
1,200,0.004,0.015,0.004
2,200,0.008,0.015,0.008
3,200,0.012,0.005,0.011
4,200,0.016,0.030,0.015
5,200,0.020,0.035,0.017
6,200,0.026,0.020,0.021
7,200,0.034,0.025,0.030
8,200,0.046,0.035,0.038
9,200,0.067,0.055,0.060


### 5.6 Model 2.6: + V3

In [20]:
evaluate(models["+V3"])

AUC          model 0.678    true probabilities 0.682
Brier score  model 0.0303   true probabilities 0.0301
mean |p_model - p_true|: 0.0092


,patients,predicted,observed,true
risk tenth,,,,
1,200,0.004,0.010,0.004
2,200,0.008,0.015,0.008
3,200,0.011,0.020,0.011
4,200,0.014,0.015,0.014
5,200,0.019,0.030,0.017
6,200,0.024,0.025,0.021
7,200,0.032,0.030,0.030
8,200,0.044,0.025,0.038
9,200,0.062,0.070,0.057
